# 10 - Search the bulletins and explain a forecast

We use the Chroma collection from notebook 09 to find bulletin passages related to a question,
and combine them with a forecast to write a short, sourced answer.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.chdir("/content/drive/MyDrive/JobAI")
except ImportError:
    pass

In [ ]:
# %pip install -q chromadb sentence-transformers pyyaml pandas "transformers==5.17.0" "peft==0.20.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" "safetensors==0.8.0" "huggingface-hub==1.31.0"
%pip install -q --no-deps "bitsandbytes==0.50.2"

from pathlib import Path
import yaml
import chromadb
from chromadb.utils import embedding_functions

REPO = Path.cwd()
config = yaml.safe_load(open(REPO / "configs" / "rag.yaml"))

client = chromadb.PersistentClient(path=str(REPO / config["vector_store"]["persist_dir"]))
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=config["embedding_candidates"][0]["id"])
collection = client.get_collection(config["vector_store"]["collection_name"], embedding_function=embedding_fn)

print(collection.count(), "chunks available to search")

## A simple search function

Given a question, return the `k` most similar chunks. `before_date` is optional: when we give
it a date, we only get chunks published on or before that date. This matters for a forecast -
we should only use bulletins that existed at the time the forecast was made, not ones from
later that would give the answer away.

In [ ]:
def search(query, k=5, before_date=None):
    # before_date is a "YYYY-MM-DD" string; Chroma can only compare numbers,
    # so we turn it into the same YYYYMMDD number used when the index was built.
    where = None
    if before_date:
        where = {"published_number": {"$lte": int(before_date.replace("-", ""))}}
    result = collection.query(query_texts=[query], n_results=k, where=where,
                              include=["documents", "metadatas", "distances"])
    hits = []
    for doc, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        hits.append({"text": doc, "similarity": round(1 - distance, 3), **meta})
    return hits

In [ ]:
for hit in search("Kuinka monta uutta avointa työpaikkaa ilmoitettiin?", k=3):
    print(f"({hit['similarity']}) {hit['title']}")
    print(" ", hit["text"][:180])

Chroma can only filter numbers, not date strings directly, which is why `search` converts `before_date` into a plain number first. Let's check the filter actually excludes later bulletins.

In [ ]:
recent = search("avoimet työpaikat", k=5, before_date="2020-01-01")
for hit in recent:
    assert hit["published"] <= "2020-01-01", hit
print("all", len(recent), "results are from before 2020")

# a document with no known date (the Statistics Finland releases) has no
# business appearing under an old cutoff either
early_hits = search("job vacancies", k=10, before_date="2015-01-01")
assert all(hit["source"] != "statfin" for hit in early_hits)

## A very small "forecast explainer"

This is where the forecasting model and the RAG meet. The forecast numbers below are a stand-in
(the real ones will come from the fine-tuned model); the point here is just to show how a
question, a forecast, and the retrieved bulletin text fit together into one answer.

In [ ]:
import json, sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(REPO))
from jobai.forecasting import input_view, prompt_messages, parse_prediction, verify_normalization
from jobai.model_runtime import selected_final_run, base_directory, load_tokenizer, load_quantized_base

# Choose a selected 12tu/12tw series from data/processed/selected_series.csv.
SERIES_ID = "12tu::Alue=MK01|Ammattiryhmä=2142|Työmarkkina-asema=SSS|contentscode=AVPAIKATYHT"
run = selected_final_run(REPO)

def prepare_forecast_inputs(series_id):
    catalog = pd.read_csv(REPO / "data/processed/selected_series.csv")
    eligible = catalog.loc[
        catalog.series_id.eq(series_id)
        & catalog.selected.astype(str).str.lower().eq("true")
        & catalog.is_forecast_target.astype(str).str.lower().eq("true")
        & catalog.table_id.isin(run["target_tables"])
    ]
    if len(eligible) != 1:
        raise ValueError("Choose one selected 12tu/12tw series; this adapter does not support other targets.")
    series = eligible.iloc[0].to_dict()
    normalization = json.loads((REPO / "data/processed/normalization_assertions.json").read_text())[series["table_id"]]
    assert normalization.get("status") == "passed", "The saved notebook 02 validation did not pass."
    # Older passed reports have no fingerprints; the forecast history is checked below.
    if normalization.get("schema_version") not in (None, 1):
        verify_normalization(REPO, [series["table_id"]])
    dimensions = json.loads(series["dimensions_json"])
    data = pd.read_csv(REPO / f"data/processed/{series['table_id']}__normalized.csv",
                       dtype=str, usecols=[*dimensions, "timeperiod_q", "value"])
    for column, value in dimensions.items():
        data = data.loc[data[column].eq(str(value))]
    history = pd.Series(pd.to_numeric(data.value, errors="coerce").to_numpy(),
                        index=pd.PeriodIndex(data.timeperiod_q, freq="Q")).sort_index()
    if history.empty or not history.index.is_unique:
        raise ValueError("The selected series has no observations or duplicate quarters; rerun notebook 02.")
    last_complete = pd.Timestamp.now().to_period("Q") - 1
    history = history.loc[history.index <= last_complete]
    if history.empty:
        raise ValueError("No complete source quarter is available for this series.")
    origin = history.index.max()
    quarters = pd.period_range(end=origin, periods=run["history_quarters"], freq="Q")
    window = history.reindex(quarters)
    if not np.isfinite(window.to_numpy()).all() or (window < 0).any():
        raise ValueError(f"Need {run['history_quarters']} consecutive quarters with valid vacancy counts through {origin}.")
    return [{**series, "origin_quarter": str(origin), "target_quarter": str(origin + h),
             "horizon_q": h, "input_values_json": json.dumps(window.tolist()),
             "last_value": float(window.iloc[-1])} for h in run["training_horizons"]]

forecast_inputs = prepare_forecast_inputs(SERIES_ID)
print("Saved model:", run["run_id"], "| training examples:", run["train_examples"])
print("Forecast origin:", forecast_inputs[0]["origin_quarter"],
      "| latest observed vacancies:", forecast_inputs[0]["last_value"])


In [ ]:
import torch
from peft import PeftModel
from transformers import set_seed
from transformers.utils import is_bitsandbytes_available

assert torch.cuda.is_available(), "Select a GPU runtime in Colab to run the saved model. No fine-tuning is needed."
assert is_bitsandbytes_available(), "Run the bitsandbytes install cell, restart the Colab session, then rerun notebook 10."
set_seed(42)
# Keep the model loaded when changing SERIES_ID and rerunning the forecast cells.
if globals().get("loaded_forecast_adapter") != run["adapter_sha256"] or "forecast_model" not in globals():
    if "forecast_model" in globals():
        del forecast_model
        torch.cuda.empty_cache()
    base_path = base_directory(REPO, run["base_model"], download=False)
    print("Using saved base model:", base_path)
    forecast_tokenizer, _ = load_tokenizer(base_path, run["base_model"], run["adapter_dir"])
    forecast_tokenizer.padding_side = "left"
    if forecast_tokenizer.pad_token_id is None:
        forecast_tokenizer.pad_token = forecast_tokenizer.eos_token
    forecast_model = PeftModel.from_pretrained(
        load_quantized_base(base_path, run["architecture"], torch.cuda.is_bf16_supported()),
        run["adapter_dir"], local_files_only=True, is_trainable=False)
    forecast_model.eval()
    loaded_forecast_adapter = run["adapter_sha256"]

prompts = [forecast_tokenizer.apply_chat_template(
    prompt_messages(row, run["prompt_schema"], run["history_quarters"]),
    tokenize=False, add_generation_prompt=True, enable_thinking=False) for row in forecast_inputs]
inputs = forecast_tokenizer(prompts, padding=True, add_special_tokens=False,
                            truncation=False, return_tensors="pt").to(forecast_model.device)
assert inputs["input_ids"].shape[1] <= run["max_seq_length"], "Forecast history exceeds the saved model's input limit."
with torch.inference_mode():
    outputs = forecast_model.generate(**inputs, max_new_tokens=64, do_sample=False,
                                      use_cache=True, pad_token_id=forecast_tokenizer.pad_token_id)
responses = forecast_tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
assert len(responses) == len(forecast_inputs)
predictions = []
for row, response in zip(forecast_inputs, responses):
    change = parse_prediction(response)
    _, scale, _, _ = input_view(row, run["history_quarters"])
    value = row["last_value"] + change * scale if change is not None else None
    if value is None or not np.isfinite(value):
        raise ValueError(f"H{row['horizon_q']} forecast could not be parsed: {response}")
    predictions.append({"run_id": run["run_id"], "series_id": row["series_id"],
                        "origin_quarter": row["origin_quarter"], "target_quarter": row["target_quarter"],
                        "horizon_q": row["horizon_q"], "last_value": row["last_value"],
                        "y_pred": max(0.0, value), "response": response})
forecasts = pd.DataFrame(predictions)
latest_value = forecast_inputs[0]["last_value"]
directions = ["rise" if p > latest_value else "fall" if p < latest_value else "unchanged" for p in forecasts.y_pred]
forecast_result = {f"forecast_{int(row.horizon_q)}q": round(float(row.y_pred), 2) for row in forecasts.itertuples()}
forecast_result["trend"] = "; ".join(f"{h}Q: {direction}" for h, direction in zip(forecasts.horizon_q, directions))
(REPO / "reports").mkdir(exist_ok=True)
forecasts.to_csv(REPO / "reports/forecast_demo_predictions.csv", index=False)
print(json.dumps(forecast_result, indent=2))
print("Forecasts for", SERIES_ID)
display(forecasts[["origin_quarter", "horizon_q", "target_quarter", "last_value", "y_pred"]].round(2))

# Source passages add context to the numeric forecasts; they do not establish causes.
cutoff = pd.Period(forecast_inputs[0]["origin_quarter"], freq="Q").end_time.strftime("%Y-%m-%d")
question = f"Job vacancies {forecast_inputs[0]['origin_quarter']}, {forecast_inputs[0]['dimensions_json']}"
hits = search(question, k=2, before_date=cutoff)
lines = [f"Official bulletin context (indexed publication dates through {cutoff}):"]
for hit in hits:
    url = hit.get("url")
    published = pd.to_datetime(hit.get("published"), errors="coerce")
    if not isinstance(url, str) or not url.startswith("https://") or pd.isna(published):
        continue
    if published.date().isoformat() > cutoff:
        continue
    lines.append(f"- [{hit['title']}]({url}) ({hit['published']}): {hit['text'][:400]}")
if len(lines) == 1:
    lines.append("No dated passage with a source link was found for this forecast.")
display(Markdown("\n\n".join(lines)))


In [ ]:
# Ask about the forecast after the preceding cell has run.
def chat_forecast(question):
    question = question.strip()
    if not question:
        raise ValueError("Enter a forecast question.")
    origin = pd.Period(forecast_inputs[0]["origin_quarter"], freq="Q")
    cutoff = origin.end_time.date().isoformat()
    recent_since = (origin - 1).start_time.date()
    hits = search(f"{question} vacancies {forecast_inputs[0]['dimensions_json']}",
                  k=8, before_date=cutoff)
    sources = []
    for hit in hits:
        published = pd.to_datetime(hit.get("published"), errors="coerce")
        url = hit.get("url")
        if (isinstance(url, str) and url.startswith("https://") and
                not pd.isna(published) and recent_since <= published.date() <= origin.end_time.date()):
            sources.append(hit)
    sources = sorted(sources, key=lambda hit: hit["published"], reverse=True)[:2]

    show_count = lambda value: f"{float(value):.2f}".rstrip("0").rstrip(".")
    values = [float(row.y_pred) for row in forecasts.itertuples()]
    forecast_text = "; ".join(f"{row.target_quarter}: {show_count(row.y_pred)}"
                              for row in forecasts.itertuples())
    facts = (f"Vacancies for this series were {show_count(forecast_inputs[0]['last_value'])} "
             f"in {origin}. Forecast: {forecast_text}.")
    path = [forecast_inputs[0]["last_value"], *values]
    moves = ["fall" if later < earlier else "rise" if later > earlier else "stay level"
             for earlier, later in zip(path, path[1:])]
    messages = [
        {"role": "system", "content":
         "Write one short sentence describing the supplied forecast direction. "
         "Do not repeat numbers, dates, series codes, citations, or source claims. "
         "Do not claim a cause for the changes."},
        {"role": "user", "content": f"Question: {question}\nDirection by horizon: {', then '.join(moves)}."},
    ]
    prompt = forecast_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = forecast_tokenizer(prompt, add_special_tokens=False,
                                return_tensors="pt").to(forecast_model.device)
    set_seed(42)
    with torch.inference_mode(), forecast_model.disable_adapter():
        output = forecast_model.generate(
            **inputs, max_new_tokens=60, do_sample=True,
            temperature=0.7, top_p=0.8, top_k=20, use_cache=True,
            pad_token_id=forecast_tokenizer.pad_token_id)
    reply = forecast_tokenizer.decode(
        output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if (not reply or any(char.isdigit() for char in reply) or
            any(word in reply.lower() for word in ("[", "http", "bulletin", "because"))):
        reply = "The projected values " + ", then ".join(moves) + "."

    if sources:
        links = "\n".join(f"- [{hit['title']}]({hit['url']}) ({hit['published']})"
                          for hit in sources)
        context = ("Recent bulletins offer broader context; they were not inputs to the "
                   f"numeric forecast and do not establish its cause.\n\n{links}")
    else:
        context = "No dated bulletin from the last two quarters was found for context."
    answer = f"{facts}\n\n{reply}\n\n{context}"
    return answer

USER_QUESTION = "What is the forecast for this series, and what do the bulletins say?"
chat_answer = chat_forecast(USER_QUESTION)
display(Markdown(chat_answer))
